# ДЗ Topic 3 — NYC TLC Yellow Taxi у PostgreSQL

**Датасет:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) —
Yellow Taxi за **липень 2026** (`yellow_tripdata_2026-07.parquet`, 3 530 109 рядків) і довідник зон
`taxi_zone_lookup.csv` (265 зон).
**Вибірка:** 100 000 випадкових рядків (`random_state=42`), 21 колонка.

Таблиці (префікс `hw3_`):

| шар | таблиця | що в ній |
|---|---|---|
| staging | `hw3_taxi_staging` | сирі рядки CSV, усі колонки `TEXT` |
| staging | `hw3_taxi_zones_staging` | сирий довідник зон, усі колонки `TEXT` |
| clean | `hw3_taxi_zones` | довідник зон |
| clean | `hw3_taxi_trips` | поїздки з потрібними типами та обмеженнями |

PostgreSQL запускається в ноутбуці через `pgserver`. pandas потрібен лише для підготовки CSV і показу результатів,
уся перевірка та аналіз зроблені SQL-запитами.

## Завдання 1. Setup і завантаження

In [3]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import pgserver
from sqlalchemy import create_engine, make_url, text

# Кластер Postgres у локальній теці; cleanup_mode="stop" — сервер зупиняється разом з kernel.
pg = pgserver.get_server(Path("pgdata").resolve(), cleanup_mode="stop")

# get_uri() повертає URI для psycopg 3; COPY робимо через psycopg2 (copy_expert) -> міняємо драйвер.
engine = create_engine(make_url(pg.get_uri()).set(drivername="postgresql+psycopg2"))

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())
    # Для локалізації часу потрібна tz-база Postgres (у Linux/Colab-збірці pgserver вона є).
    print("Time zone America/New_York available:", conn.execute(text(
        "SELECT EXISTS (SELECT 1 FROM pg_timezone_names WHERE name = 'America/New_York')"
    )).scalar_one())

AssertionError: 

In [ ]:
pd.set_option("display.max_columns", None)


def run(sql):
    """DDL/DML без результату (в одній транзакції)."""
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)


def show(sql):
    """SELECT-запит -> DataFrame."""
    return pd.read_sql(text(sql), engine)


def copy_csv(table, csv_path):
    """CSV -> таблиця через COPY FROM STDIN (psycopg2 copy_expert)."""
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur, open(csv_path, encoding="utf-8", newline="") as f:
            cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')", f)
        raw_conn.commit()
    finally:
        raw_conn.close()

### Завантаження даних і формування вибірки

Місячний Parquet і довідник зон завантажуються з офіційного сайту TLC
([TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)); якщо файл уже є — не завантажується повторно.
З Parquet беремо 21 колонку, випадково вибираємо 100 000 рядків (`random_state=42`) і зберігаємо в CSV.

Час у Parquet **локальний** (без зони) — це час Нью-Йорка. У CSV він іде як текст, а зону додаємо вже в SQL.

In [ ]:
PARQUET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-07.parquet"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
PARQUET_PATH = Path("yellow_tripdata_2026-07.parquet")
CSV_PATH = Path("yellow_tripdata_2026-07_sample.csv")
ZONES_PATH = Path("taxi_zone_lookup.csv")

for url, path in [(PARQUET_URL, PARQUET_PATH), (ZONES_URL, ZONES_PATH)]:
    if not path.exists():
        print("Завантажую", url)
        urlretrieve(url, path)
    print(path, f"{path.stat().st_size / 1e6:.2f} MB")

In [ ]:
COLUMNS = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "cbd_congestion_fee", "request_source",
]

df = pd.read_parquet(PARQUET_PATH, columns=COLUMNS)
sample = df.sample(n=100_000, random_state=42).sort_index()
sample.columns = [c.lower() for c in COLUMNS]
sample.to_csv(CSV_PATH, index=False)

print(f"Parquet: {len(df):,} рядків -> вибірка: {len(sample):,} рядків ({CSV_PATH})")
sample.dtypes

### Staging-таблиці та `COPY FROM STDIN`

Усі сирі колонки мають тип `TEXT` (суфікс `_raw`) — `COPY` не впаде на «брудному» значенні,
а приведення типів і очищення виконуються контрольовано на наступному кроці.

In [ ]:
run("""
    DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;
    DROP TABLE IF EXISTS hw3_taxi_zones CASCADE;
    DROP TABLE IF EXISTS hw3_taxi_staging CASCADE;
    DROP TABLE IF EXISTS hw3_taxi_zones_staging CASCADE;

    CREATE TABLE hw3_taxi_staging (
        vendorid_raw              TEXT,
        tpep_pickup_datetime_raw  TEXT,
        tpep_dropoff_datetime_raw TEXT,
        passenger_count_raw       TEXT,
        trip_distance_raw         TEXT,
        ratecodeid_raw            TEXT,
        store_and_fwd_flag_raw    TEXT,
        pulocationid_raw          TEXT,
        dolocationid_raw          TEXT,
        payment_type_raw          TEXT,
        fare_amount_raw           TEXT,
        extra_raw                 TEXT,
        mta_tax_raw               TEXT,
        tip_amount_raw            TEXT,
        tolls_amount_raw          TEXT,
        improvement_surcharge_raw TEXT,
        total_amount_raw          TEXT,
        congestion_surcharge_raw  TEXT,
        airport_fee_raw           TEXT,
        cbd_congestion_fee_raw    TEXT,
        request_source_raw        TEXT
    );

    CREATE TABLE hw3_taxi_zones_staging (
        locationid_raw   TEXT,
        borough_raw      TEXT,
        zone_raw         TEXT,
        service_zone_raw TEXT
    );
""")

In [ ]:
copy_csv("hw3_taxi_staging", CSV_PATH)
copy_csv("hw3_taxi_zones_staging", ZONES_PATH)

#### COPY: рядків у staging

In [ ]:
show("""
    SELECT 'hw3_taxi_staging' AS table_name, COUNT(*) AS loaded_rows FROM hw3_taxi_staging
    UNION ALL
    SELECT 'hw3_taxi_zones_staging', COUNT(*) FROM hw3_taxi_zones_staging;
""")

**Висновок.** Завантажено всі 100 000 рядків вибірки та 265 зон довідника — втрат на імпорті немає.

#### Staging: приклад сирих рядків

In [ ]:
show("""
    SELECT * FROM hw3_taxi_staging LIMIT 3;
""")

**Висновок.** Дати записані як текст `YYYY-MM-DD HH:MM:SS` без часового поясу, числа — як `1.0`, `10.7`.
Порожні значення (наприклад, `request_source`) завантажились як `NULL` (`None`).

## Завдання 2. Staging і DDL

### Колонки: зміст, формат у джерелі, тип і очищення

Для всіх полів: `NULLIF(TRIM(x), '')` — прибираємо пробіли, порожній рядок стає `NULL`, потім приводимо до типу.

| колонка(и) | що означає | у CSV | тип | очищення / обмеження |
|---|---|---|---|---|
| `trip_id` | технічний ключ поїздки | — | `BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY` | у джерелі немає ID поїздки, тому генеруємо свій |
| `vendor_id` | компанія, що передала запис (1, 2, 6, 7) | `"2"` | `SMALLINT NOT NULL` | маленький код, без нього запис не має джерела |
| `pickup_at`, `dropoff_at` | час посадки / висадки | `"2026-07-01 00:25:36"` (час NY без зони) | `TIMESTAMPTZ NOT NULL` | `::TIMESTAMP AT TIME ZONE 'America/New_York'`; `CHECK (dropoff_at >= pickup_at)` |
| `passenger_count` | кількість пасажирів | `"1.0"` | `SMALLINT` | `::NUMERIC::SMALLINT`, бо в CSV дробовий запис; `NULL` дозволено; `CHECK (>= 0)` |
| `trip_distance` | відстань, милі | `"1.78"` | `NUMERIC(10,2) NOT NULL` | `CHECK (trip_distance >= 0)` |
| `ratecode_id` | тариф (1 звичайний, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 домовлений, 99 невідомо) | `"1.0"` | `SMALLINT` | `::NUMERIC::SMALLINT`; 99 залишаємо, але позначаємо прапорцем |
| `store_and_fwd` | запис було збережено в машині й надіслано пізніше | `"Y"` / `"N"` | `BOOLEAN` | `Y → TRUE`, `N → FALSE`, інше → `NULL` |
| `pu_location_id`, `do_location_id` | зона посадки / висадки | `"233"` | `SMALLINT NOT NULL REFERENCES hw3_taxi_zones` | це коди категорій; FK перевіряє, що зона є в довіднику |
| `payment_type` | спосіб оплати (0 Flex Fare, 1 картка, 2 готівка, 3 без оплати, 4 спір, 5 невідомо, 6 анульовано) | `"1"` | `SMALLINT NOT NULL` | `CHECK (payment_type BETWEEN 0 AND 6)` |
| `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `airport_fee`, `cbd_congestion_fee` | частини чека і загальна сума, $ | `"10.7"` | `NUMERIC(10,2)`; `fare_amount`, `total_amount` — `NOT NULL` | гроші з точністю до центів; від'ємні суми не відкидаємо (це повернення коштів), а позначаємо прапорцем |
| `request_source` | код джерела замовлення (нове поле) | `"HV0003"` | `TEXT` | обмеження довжини немає, тому `TEXT`, а не `VARCHAR(255)` |

Довідник `hw3_taxi_zones`: `location_id SMALLINT PRIMARY KEY` (1–265), інші колонки — `TEXT`;
значення `'N/A'` і `'Unknown'` означають пропуск, тому замінюємо їх на `NULL`.

Окремої колонки-дати в джерелі немає; дату отримуємо з `pickup_at` через `::DATE` (див. прапорці та Q10).

In [ ]:
run("""
    CREATE TABLE hw3_taxi_zones (
        location_id  SMALLINT PRIMARY KEY CHECK (location_id BETWEEN 1 AND 265),
        borough      TEXT,
        zone         TEXT,
        service_zone TEXT
    );

    INSERT INTO hw3_taxi_zones (location_id, borough, zone, service_zone)
    SELECT
        TRIM(locationid_raw)::SMALLINT,
        NULLIF(NULLIF(TRIM(borough_raw), 'N/A'), 'Unknown'),
        NULLIF(NULLIF(TRIM(zone_raw), 'N/A'), 'Unknown'),
        NULLIF(NULLIF(TRIM(service_zone_raw), 'N/A'), 'Unknown')
    FROM hw3_taxi_zones_staging;
""")

In [ ]:
run("""
    CREATE TABLE hw3_taxi_trips (
        trip_id               BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
        vendor_id             SMALLINT NOT NULL,
        pickup_at             TIMESTAMPTZ NOT NULL,
        dropoff_at            TIMESTAMPTZ NOT NULL,
        passenger_count       SMALLINT
            CONSTRAINT chk_passenger_count CHECK (passenger_count IS NULL OR passenger_count >= 0),
        trip_distance         NUMERIC(10, 2) NOT NULL
            CONSTRAINT chk_trip_distance CHECK (trip_distance >= 0),
        ratecode_id           SMALLINT,
        store_and_fwd         BOOLEAN,
        pu_location_id        SMALLINT NOT NULL REFERENCES hw3_taxi_zones (location_id),
        do_location_id        SMALLINT NOT NULL REFERENCES hw3_taxi_zones (location_id),
        payment_type          SMALLINT NOT NULL
            CONSTRAINT chk_payment_type CHECK (payment_type BETWEEN 0 AND 6),
        fare_amount           NUMERIC(10, 2) NOT NULL,
        extra                 NUMERIC(10, 2),
        mta_tax               NUMERIC(10, 2),
        tip_amount            NUMERIC(10, 2),
        tolls_amount          NUMERIC(10, 2),
        improvement_surcharge NUMERIC(10, 2),
        total_amount          NUMERIC(10, 2) NOT NULL,
        congestion_surcharge  NUMERIC(10, 2),
        airport_fee           NUMERIC(10, 2),
        cbd_congestion_fee    NUMERIC(10, 2),
        request_source        TEXT,
        CONSTRAINT chk_dropoff_after_pickup CHECK (dropoff_at >= pickup_at)
    );

    WITH parsed AS (
        SELECT
            NULLIF(TRIM(vendorid_raw), '')::SMALLINT                                        AS vendor_id,
            NULLIF(TRIM(tpep_pickup_datetime_raw), '')::TIMESTAMP AT TIME ZONE 'America/New_York'  AS pickup_at,
            NULLIF(TRIM(tpep_dropoff_datetime_raw), '')::TIMESTAMP AT TIME ZONE 'America/New_York' AS dropoff_at,
            NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT                        AS passenger_count,
            NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(10, 2)                             AS trip_distance,
            NULLIF(TRIM(ratecodeid_raw), '')::NUMERIC::SMALLINT                             AS ratecode_id,
            CASE UPPER(TRIM(store_and_fwd_flag_raw)) WHEN 'Y' THEN TRUE WHEN 'N' THEN FALSE END AS store_and_fwd,
            NULLIF(TRIM(pulocationid_raw), '')::SMALLINT                                    AS pu_location_id,
            NULLIF(TRIM(dolocationid_raw), '')::SMALLINT                                    AS do_location_id,
            NULLIF(TRIM(payment_type_raw), '')::SMALLINT                                    AS payment_type,
            NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10, 2)                               AS fare_amount,
            NULLIF(TRIM(extra_raw), '')::NUMERIC(10, 2)                                     AS extra,
            NULLIF(TRIM(mta_tax_raw), '')::NUMERIC(10, 2)                                   AS mta_tax,
            NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10, 2)                                AS tip_amount,
            NULLIF(TRIM(tolls_amount_raw), '')::NUMERIC(10, 2)                              AS tolls_amount,
            NULLIF(TRIM(improvement_surcharge_raw), '')::NUMERIC(10, 2)                     AS improvement_surcharge,
            NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10, 2)                              AS total_amount,
            NULLIF(TRIM(congestion_surcharge_raw), '')::NUMERIC(10, 2)                      AS congestion_surcharge,
            NULLIF(TRIM(airport_fee_raw), '')::NUMERIC(10, 2)                               AS airport_fee,
            NULLIF(TRIM(cbd_congestion_fee_raw), '')::NUMERIC(10, 2)                        AS cbd_congestion_fee,
            NULLIF(TRIM(request_source_raw), '')                                            AS request_source
        FROM hw3_taxi_staging
    )
    INSERT INTO hw3_taxi_trips (
        vendor_id, pickup_at, dropoff_at, passenger_count, trip_distance,
        ratecode_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
        fare_amount, extra, mta_tax, tip_amount, tolls_amount,
        improvement_surcharge, total_amount, congestion_surcharge,
        airport_fee, cbd_congestion_fee, request_source
    )
    SELECT
        vendor_id, pickup_at, dropoff_at, passenger_count, trip_distance,
        ratecode_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
        fare_amount, extra, mta_tax, tip_amount, tolls_amount,
        improvement_surcharge, total_amount, congestion_surcharge,
        airport_fee, cbd_congestion_fee, request_source
    FROM parsed
    WHERE vendor_id IS NOT NULL
      AND pickup_at IS NOT NULL
      AND dropoff_at IS NOT NULL
      AND dropoff_at >= pickup_at
      AND trip_distance >= 0
      AND (passenger_count IS NULL OR passenger_count >= 0)
      AND pu_location_id BETWEEN 1 AND 265
      AND do_location_id BETWEEN 1 AND 265
      AND payment_type BETWEEN 0 AND 6
      AND fare_amount IS NOT NULL
      AND total_amount IS NOT NULL;
""")

Умова `WHERE` в `INSERT … SELECT` повторює обмеження таблиці: якщо рядок їх порушує, він просто не потрапить
у clean-таблицю, а не зламає весь імпорт (перевірка — у 3.1).

#### Перевірка локалізації часу (UTC vs New York)

In [ ]:
show("""
    WITH l AS (
        SELECT
            tpep_pickup_datetime_raw AS pickup_raw,
            tpep_pickup_datetime_raw::TIMESTAMP AT TIME ZONE 'America/New_York' AS pickup_at
        FROM hw3_taxi_staging
        LIMIT 3
    )
    SELECT
        pickup_raw,
        pickup_at AT TIME ZONE 'UTC'              AS pickup_utc,
        pickup_at AT TIME ZONE 'America/New_York' AS pickup_ny_local
    FROM l;
""")

**Висновок.** Сирий час `00:25:36` сприйнято як час Нью-Йорка: у UTC це `04:25:36` (влітку NY = UTC−4),
а при переведенні назад отримуємо початкове значення. Отже, часовий пояс задано правильно.

Додаткові колонки для аналізу: тривалість у хвилинах, година й день тижня (за **часом Нью-Йорка**) і середня швидкість.

In [ ]:
run("""
    ALTER TABLE hw3_taxi_trips
        ADD COLUMN IF NOT EXISTS trip_duration_min NUMERIC(10, 2),
        ADD COLUMN IF NOT EXISTS pickup_hour       SMALLINT,
        ADD COLUMN IF NOT EXISTS pickup_dow        SMALLINT,
        ADD COLUMN IF NOT EXISTS avg_speed_mph     NUMERIC(10, 2);

    UPDATE hw3_taxi_trips
    SET
        trip_duration_min = ROUND(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 60, 2),
        pickup_hour       = EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York'),
        pickup_dow        = EXTRACT(ISODOW FROM pickup_at AT TIME ZONE 'America/New_York'),
        avg_speed_mph     = CASE
                                WHEN dropoff_at > pickup_at
                                THEN ROUND(trip_distance / (EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600), 2)
                            END;
""")

## Завдання 3. Data-quality audit

### 3.1. Контроль кількості рядків

#### Рядків у staging vs clean

In [ ]:
show("""
    SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_taxi_staging
    UNION ALL
    SELECT 'clean' AS layer, COUNT(*) AS n_rows FROM hw3_taxi_trips;
""")

#### Скільки staging-рядків порушує кожне правило фільтра

In [ ]:
show("""
    SELECT
        COUNT(*) FILTER (WHERE vendorid_raw IS NULL OR tpep_pickup_datetime_raw IS NULL
                            OR tpep_dropoff_datetime_raw IS NULL)                      AS missing_key_fields,
        COUNT(*) FILTER (WHERE tpep_dropoff_datetime_raw::TIMESTAMP
                             < tpep_pickup_datetime_raw::TIMESTAMP)                    AS dropoff_before_pickup,
        COUNT(*) FILTER (WHERE trip_distance_raw::NUMERIC < 0)                         AS negative_distance,
        COUNT(*) FILTER (WHERE passenger_count_raw::NUMERIC < 0)                       AS negative_passengers,
        COUNT(*) FILTER (WHERE pulocationid_raw::INTEGER NOT BETWEEN 1 AND 265
                            OR dolocationid_raw::INTEGER NOT BETWEEN 1 AND 265)        AS bad_location,
        COUNT(*) FILTER (WHERE payment_type_raw::INTEGER NOT BETWEEN 0 AND 6)          AS bad_payment_type,
        COUNT(*) FILTER (WHERE fare_amount_raw IS NULL OR total_amount_raw IS NULL)    AS missing_amounts
    FROM hw3_taxi_staging;
""")

**Висновок.** У staging і clean однаково по **100 000** рядків: жоден рядок не порушив правил фільтра
(обов'язкові поля не порожні, `dropoff_at >= pickup_at`, дистанція й пасажири не від'ємні, зони 1–265, `payment_type` 0–6).
Дивні, але можливі значення (нульова дистанція, від'ємні суми) не видаляємо, а позначаємо прапорцями нижче.

### 3.2. Перевірка дублікатів

#### Точні дублікати у staging

In [ ]:
show("""
    SELECT
        (SELECT COUNT(*) FROM hw3_taxi_staging) AS total_rows,
        (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM hw3_taxi_staging) AS d) AS distinct_rows;
""")

**Висновок.** Повністю однакових рядків немає.

#### Потенційні дублікати за business key

ID поїздки в даних TLC немає, тому як business key беру
`(vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, fare_amount)` — той самий перевізник,
той самий час, маршрут і тариф. Така перевірка знаходить лише **потенційні** дублікати.

In [ ]:
show("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, fare_amount))
            AS distinct_business_keys,
        COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, fare_amount))
            AS duplicate_rows
    FROM hw3_taxi_trips;
""")

**Висновок.** 41 рядок (0.04 %) має такий самий business key, як інший рядок.

#### Групи потенційних дублікатів

In [ ]:
show("""
    SELECT
        vendor_id,
        pickup_at AT TIME ZONE 'America/New_York' AS pickup_ny,
        pu_location_id,
        do_location_id,
        fare_amount,
        COUNT(*)          AS n_rows,
        MIN(total_amount) AS min_total,
        MAX(total_amount) AS max_total,
        MIN(payment_type) AS min_payment_type,
        MAX(payment_type) AS max_payment_type
    FROM hw3_taxi_trips
    GROUP BY vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, fare_amount
    HAVING COUNT(*) > 1
    ORDER BY n_rows DESC, pickup_at
    LIMIT 10;
""")

**Висновок.** Усі такі пари — від vendor 1: одна й та сама поїздка записана двічі, з `payment_type` 1 (картка)
і 0 (Flex Fare), часто з різним `total_amount`. Яка з двох копій правильна, з даних не зрозуміло,
тому не видаляю їх, а залишаю для перевірки джерела.

### 3.3. Null-rate і спеціальні маркери

#### Null-rate по колонках clean, %

In [ ]:
show("""
    SELECT
        COUNT(*) AS total_rows,
        ROUND(COUNT(*) FILTER (WHERE passenger_count      IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_passenger_count,
        ROUND(COUNT(*) FILTER (WHERE ratecode_id          IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_ratecode_id,
        ROUND(COUNT(*) FILTER (WHERE store_and_fwd        IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_store_and_fwd,
        ROUND(COUNT(*) FILTER (WHERE congestion_surcharge IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_congestion,
        ROUND(COUNT(*) FILTER (WHERE airport_fee          IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_airport_fee,
        ROUND(COUNT(*) FILTER (WHERE cbd_congestion_fee   IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_cbd_fee,
        ROUND(COUNT(*) FILTER (WHERE tip_amount           IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_tip_amount,
        ROUND(COUNT(*) FILTER (WHERE request_source       IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_request_source
    FROM hw3_taxi_trips;
""")

**Висновок.** У п'яти колонках (`passenger_count`, `ratecode_id`, `store_and_fwd`, `congestion_surcharge`, `airport_fee`)
однаково **27.58 %** пропусків, у `request_source` — 72.43 %, а `cbd_congestion_fee` і `tip_amount` заповнені повністю.
Але 0 % пропусків ще не означає, що дані хороші: при оплаті готівкою `tip_amount = 0`, бо чайові просто не записуються.

#### Спеціальні маркери пропуску в сирих даних

In [ ]:
show("""
    SELECT 'NULL (порожнє поле CSV) у passenger_count' AS marker,
           COUNT(*) FILTER (WHERE passenger_count_raw IS NULL) AS n_rows
    FROM hw3_taxi_staging
    UNION ALL
    SELECT 'порожній рядок / пробіли у текстових полях',
           COUNT(*) FILTER (WHERE TRIM(passenger_count_raw) = '' OR TRIM(ratecodeid_raw) = ''
                               OR TRIM(store_and_fwd_flag_raw) = '' OR TRIM(request_source_raw) = '')
    FROM hw3_taxi_staging
    UNION ALL
    SELECT 'текстові маркери ?, unknown, NA, null',
           COUNT(*) FILTER (WHERE LOWER(TRIM(request_source_raw)) IN ('?', 'unknown', 'na', 'n/a', 'null')
                               OR LOWER(TRIM(store_and_fwd_flag_raw)) IN ('?', 'unknown', 'na', 'n/a', 'null'))
    FROM hw3_taxi_staging
    UNION ALL
    SELECT 'payment_type = 0 (Flex Fare, без деталей)',   COUNT(*) FILTER (WHERE payment_type_raw = '0') FROM hw3_taxi_staging
    UNION ALL
    SELECT 'payment_type = 5 (Unknown)',                  COUNT(*) FILTER (WHERE payment_type_raw = '5') FROM hw3_taxi_staging
    UNION ALL
    SELECT 'RatecodeID = 99 (Null/unknown)',               COUNT(*) FILTER (WHERE ratecodeid_raw::NUMERIC = 99) FROM hw3_taxi_staging
    UNION ALL
    SELECT 'LocationID = 264 (Unknown)',                  COUNT(*) FILTER (WHERE pulocationid_raw = '264' OR dolocationid_raw = '264') FROM hw3_taxi_staging
    UNION ALL
    SELECT 'LocationID = 265 (Outside of NYC)',           COUNT(*) FILTER (WHERE pulocationid_raw = '265' OR dolocationid_raw = '265') FROM hw3_taxi_staging
    UNION ALL
    SELECT 'store_and_fwd_flag не Y/N',                   COUNT(*) FILTER (WHERE store_and_fwd_flag_raw NOT IN ('Y', 'N')) FROM hw3_taxi_staging
    UNION ALL
    SELECT 'zones: ''N/A'' / ''Unknown'' у довіднику',     COUNT(*) FILTER (WHERE 'N/A' IN (borough_raw, zone_raw, service_zone_raw)
                                                                          OR 'Unknown' IN (borough_raw, zone_raw, service_zone_raw))
    FROM hw3_taxi_zones_staging;
""")

**Висновок.** Текстових маркерів на кшталт `?`, `unknown` або порожнього рядка немає, бо CSV я зробив сам із Parquet.
Зате «невідомо» записано числовими кодами: `RatecodeID = 99` (1 456 рядків), зона 264 Unknown (178).
Зона 265 «Outside of NYC» (644) — це не пропуск, а нормальне значення. У довіднику зон 2 рядки мали `'N/A'` / `'Unknown'`.

#### Чи збігаються пропуски? (passenger_count / ratecode_id / payment_type)

In [ ]:
show("""
    SELECT
        passenger_count IS NULL AS passenger_is_null,
        ratecode_id IS NULL     AS ratecode_is_null,
        payment_type,
        COUNT(*) AS n
    FROM hw3_taxi_trips
    GROUP BY 1, 2, 3
    ORDER BY n DESC;
""")

**Висновок.** Пропуски не випадкові: `passenger_count` і `ratecode_id` порожні рівно в 27 582 рядках з `payment_type = 0`
(Flex Fare), і тільки в них. Тобто це окремий тип записів, у якому ці поля просто не заповнюються.
Заповнювати їх середнім не варто.

#### Домени категоріальних кодів

In [ ]:
show("""
    SELECT 'vendor_id' AS col, vendor_id::TEXT AS value, COUNT(*) AS n FROM hw3_taxi_trips GROUP BY vendor_id
    UNION ALL
    SELECT 'ratecode_id', ratecode_id::TEXT, COUNT(*) FROM hw3_taxi_trips GROUP BY ratecode_id
    UNION ALL
    SELECT 'passenger_count', passenger_count::TEXT, COUNT(*) FROM hw3_taxi_trips GROUP BY passenger_count
    UNION ALL
    SELECT 'request_source', request_source, COUNT(*) FROM hw3_taxi_trips GROUP BY request_source
    ORDER BY col, n DESC;
""")

**Висновок.** Є 234 поїздки з `passenger_count = 0` — так не буває, це помилка введення. Vendor 7 і 6 трапляються рідко
(1 250 і 223 рядки). `request_source` заповнений лише в чверті рядків кодами `HV0003`, `A`, `EH0004`, `CC`,
і що вони означають, у документації не пояснено.

### Прапорці якості

Для кожного правила додаємо окрему колонку `BOOLEAN`, а `is_valid = TRUE`, якщо жоден прапорець не спрацював.
Рядки не видаляємо — їх можна виключити з навчання моделі або перевірити окремо.

| прапорець | умова |
|---|---|
| `flag_outside_month` | посадка не в липні 2026 (за часом NY) |
| `flag_zero_duration` | `dropoff = pickup` |
| `flag_long_duration` | поїздка довша 6 годин |
| `flag_zero_distance` | дистанція = 0 |
| `flag_negative_amount` | від'ємний `fare_amount` або `total_amount` |
| `flag_zero_passengers` | `passenger_count = 0` |
| `flag_unknown_code` | `ratecode_id = 99` або зона 264 (Unknown) |
| `flag_too_fast` | середня швидкість > 80 mph |

`passenger_count`, `ratecode_id` і `avg_speed_mph` можуть бути `NULL`, тому порівняння загорнуто в `COALESCE(…, FALSE)`.

In [ ]:
run("""
    ALTER TABLE hw3_taxi_trips
        ADD COLUMN IF NOT EXISTS flag_outside_month   BOOLEAN,
        ADD COLUMN IF NOT EXISTS flag_zero_duration   BOOLEAN,
        ADD COLUMN IF NOT EXISTS flag_long_duration   BOOLEAN,
        ADD COLUMN IF NOT EXISTS flag_zero_distance   BOOLEAN,
        ADD COLUMN IF NOT EXISTS flag_negative_amount BOOLEAN,
        ADD COLUMN IF NOT EXISTS flag_zero_passengers BOOLEAN,
        ADD COLUMN IF NOT EXISTS flag_unknown_code    BOOLEAN,
        ADD COLUMN IF NOT EXISTS flag_too_fast        BOOLEAN,
        ADD COLUMN IF NOT EXISTS is_valid             BOOLEAN;

    UPDATE hw3_taxi_trips
    SET
        flag_outside_month   = (pickup_at AT TIME ZONE 'America/New_York')::DATE NOT BETWEEN '2026-07-01' AND '2026-07-31',
        flag_zero_duration   = dropoff_at = pickup_at,
        flag_long_duration   = dropoff_at - pickup_at > INTERVAL '6 hours',
        flag_zero_distance   = trip_distance = 0,
        flag_negative_amount = fare_amount < 0 OR total_amount < 0,
        flag_zero_passengers = COALESCE(passenger_count = 0, FALSE),
        flag_unknown_code    = COALESCE(ratecode_id = 99, FALSE) OR pu_location_id = 264 OR do_location_id = 264,
        flag_too_fast        = COALESCE(avg_speed_mph > 80, FALSE);

    UPDATE hw3_taxi_trips
    SET is_valid = NOT (flag_outside_month OR flag_zero_duration OR flag_long_duration OR flag_zero_distance
                        OR flag_negative_amount OR flag_zero_passengers OR flag_unknown_code OR flag_too_fast);
""")

#### Скільки рядків має кожен прапорець

In [ ]:
show("""
    SELECT 'flag_zero_distance' AS flag, COUNT(*) FILTER (WHERE flag_zero_distance) AS n_rows FROM hw3_taxi_trips
    UNION ALL
    SELECT 'flag_unknown_code', COUNT(*) FILTER (WHERE flag_unknown_code) FROM hw3_taxi_trips
    UNION ALL
    SELECT 'flag_zero_duration', COUNT(*) FILTER (WHERE flag_zero_duration) FROM hw3_taxi_trips
    UNION ALL
    SELECT 'flag_negative_amount', COUNT(*) FILTER (WHERE flag_negative_amount) FROM hw3_taxi_trips
    UNION ALL
    SELECT 'flag_zero_passengers', COUNT(*) FILTER (WHERE flag_zero_passengers) FROM hw3_taxi_trips
    UNION ALL
    SELECT 'flag_too_fast', COUNT(*) FILTER (WHERE flag_too_fast) FROM hw3_taxi_trips
    UNION ALL
    SELECT 'flag_long_duration', COUNT(*) FILTER (WHERE flag_long_duration) FROM hw3_taxi_trips
    UNION ALL
    SELECT 'flag_outside_month', COUNT(*) FILTER (WHERE flag_outside_month) FROM hw3_taxi_trips
    ORDER BY n_rows DESC;
""")

**Висновок.** Найчастіша проблема — нульова дистанція (3 663 рядки, 3.7 %), далі невідомий тариф/зона (1 634)
і нульова тривалість (1 256).

#### Скільки рядків валідні

In [ ]:
show("""
    SELECT
        is_valid,
        COUNT(*) AS n_rows,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
    FROM hw3_taxi_trips
    GROUP BY is_valid
    ORDER BY is_valid DESC;
""")

**Висновок.** 92.92 % рядків без жодного прапорця, у 7.08 % є хоча б один.

#### Частка проблемних рядків за vendor_id

In [ ]:
show("""
    SELECT
        vendor_id,
        COUNT(*) AS n_trips,
        COUNT(*) FILTER (WHERE NOT is_valid) AS n_invalid,
        ROUND(COUNT(*) FILTER (WHERE NOT is_valid) * 100.0 / COUNT(*), 2) AS pct_invalid
    FROM hw3_taxi_trips
    GROUP BY vendor_id
    ORDER BY pct_invalid DESC;
""")

**Висновок.** Якість сильно залежить від перевізника: у vendor 7 усі 1 250 рядків проблемні (у нього час висадки
завжди дорівнює часу посадки), у vendor 1 — 12.1 %, у vendor 2 — 4.7 %, у vendor 6 — жодного.

#### Розбіжність total_amount vs сума компонентів (vendor × payment_type)

`IS DISTINCT FROM` — NULL-безпечне порівняння: рядок з `NULL` в одній зі сторін теж вважається розбіжністю.

In [ ]:
show("""
    WITH d AS (
        SELECT
            vendor_id,
            payment_type,
            total_amount,
            fare_amount + extra + mta_tax + tip_amount + tolls_amount
                + improvement_surcharge + COALESCE(congestion_surcharge, 0)
                + COALESCE(airport_fee, 0) + COALESCE(cbd_congestion_fee, 0) AS components_sum
        FROM hw3_taxi_trips
    )
    SELECT
        vendor_id,
        payment_type,
        total_amount - components_sum AS diff,
        COUNT(*) AS n_rows
    FROM d
    WHERE total_amount IS DISTINCT FROM components_sum
    GROUP BY vendor_id, payment_type, total_amount - components_sum
    HAVING COUNT(*) >= 100
    ORDER BY n_rows DESC;
""")

**Висновок.** `total_amount` не дорівнює сумі складових приблизно в 40 % рядків (39 532), і різниця майже завжди
однакова: у Flex Fare vendor 2 — рівно +2.50 $ (17 064 рядки), у vendor 1 — −3.25 $ або −2.50 $.
Схоже, різні перевізники по-різному записують збори. Це не помилка окремих рядків, тому не позначаю їх як невалідні,
але при побудові ознак це треба врахувати.

## Завдання 4. Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальної колонки

#### Розподіл payment_type

In [ ]:
show("""
    SELECT
        payment_type,
        COUNT(*) AS n_rows,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
    FROM hw3_taxi_trips
    GROUP BY payment_type
    ORDER BY n_rows DESC, payment_type
    LIMIT 10;
""")

**Висновок.** Категорії дуже нерівні: картка — 61.8 %, Flex Fare — 27.6 %, готівка — 9.7 %, а спір (0.62 %)
і «без оплати» (0.31 %) трапляються рідко; кодів 5 і 6 у вибірці немає. Для моделі рідкісні коди краще
об'єднати в «інше», а Flex Fare розглядати окремо, бо там бракує частини полів.

#### Розподіл ratecode_id

In [ ]:
show("""
    SELECT
        ratecode_id,
        COUNT(*) AS n_rows,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
    FROM hw3_taxi_trips
    GROUP BY ratecode_id
    ORDER BY n_rows DESC, ratecode_id
    LIMIT 10;
""")

**Висновок.** Звичайний тариф — 67 %, `NULL` (Flex Fare) — 27.6 %. Аеропортні й міжміські тарифи (2–5) разом ≈ 4 %,
але саме це найдорожчі поїздки. Невідомий тариф `99` — 1.46 %.

### 4.2. Базова статистика числової колонки

#### Базова статистика fare_amount

In [ ]:
show("""
    SELECT
        COUNT(fare_amount) AS n_non_null,
        MIN(fare_amount) AS min_value,
        MAX(fare_amount) AS max_value,
        ROUND(AVG(fare_amount)::NUMERIC, 2) AS avg_value,
        ROUND(STDDEV_SAMP(fare_amount)::NUMERIC, 2) AS sd_value
    FROM hw3_taxi_trips;
""")

**Висновок.** `fare_amount` від −200 $ (повернення коштів при спорі) до 553 $ (домовлений тариф при нульовій дистанції);
середнє 21.02 $, стандартне відхилення 18.46 $.

#### Середнє та медіана (валідні поїздки)

In [ ]:
show("""
    SELECT
        ROUND(AVG(fare_amount), 2)                                  AS avg_fare,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY fare_amount)    AS median_fare,
        ROUND(AVG(trip_distance), 2)                                AS avg_distance,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_distance)  AS median_distance,
        MAX(trip_distance)                                          AS max_distance
    FROM hw3_taxi_trips
    WHERE is_valid;
""")

**Висновок.** Середнє помітно більше за медіану (fare 20.98 $ проти 15.60 $, дистанція 3.55 проти 1.99 милі):
є небагато дуже довгих і дорогих поїздок, які «тягнуть» середнє вгору.

#### Гістограма trip_distance (по 1 милі; 20 = 20 миль і більше)

In [ ]:
show("""
    SELECT
        LEAST(FLOOR(trip_distance), 20) AS miles_from,
        COUNT(*) AS n_trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM hw3_taxi_trips
    WHERE is_valid
    GROUP BY 1
    ORDER BY 1;
""")

**Висновок.** Половина поїздок коротша за 2 милі. Є другий невеликий пік на 16–18 милях — це поїздки в аеропорт JFK і назад.

### 4.3. Outlier-check

Правило 3σ для `total_amount` і доменна перевірка сум. Розподіл сум несиметричний, тому 3σ тут показує лише
**кандидатів** на перевірку, а не обов'язково помилки.

#### Правило 3σ для total_amount (топ-20)

In [ ]:
show("""
    WITH stats AS (
        SELECT
            AVG(total_amount) AS mean_value,
            STDDEV_SAMP(total_amount) AS sd_value
        FROM hw3_taxi_trips
        WHERE total_amount IS NOT NULL
    )
    SELECT
        t.trip_id,
        t.vendor_id,
        t.ratecode_id,
        t.pu_location_id,
        t.do_location_id,
        t.trip_distance,
        t.trip_duration_min,
        t.total_amount,
        ROUND(((t.total_amount - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC, 2) AS z_score
    FROM hw3_taxi_trips AS t
         CROSS JOIN stats AS s
    WHERE t.total_amount IS NOT NULL
      AND s.sd_value > 0
      AND ABS(t.total_amount - s.mean_value) > 3 * s.sd_value
    ORDER BY ABS(t.total_amount - s.mean_value) DESC
    LIMIT 20;
""")

**Висновок.** Найбільші значення (350–554 $) — це в основному домовлений тариф (`ratecode_id = 5`) або тариф 4 з висадкою
в зоні 265 «Outside of NYC»: довгі поїздки за місто на 50–130 миль, тобто рідкісні, але справжні.
Підозріло виглядають рядки з дистанцією 0 і тривалістю менше хвилини за 350–554 $ — ціну вписано, а самої поїздки немає.

#### Скільки 3σ-викидів total_amount і за яким ratecode_id

In [ ]:
show("""
    WITH stats AS (
        SELECT AVG(total_amount) AS mean_value, STDDEV_SAMP(total_amount) AS sd_value
        FROM hw3_taxi_trips
    )
    SELECT
        t.ratecode_id,
        COUNT(*) AS n_outliers,
        ROUND(AVG(t.trip_distance), 2) AS avg_distance,
        ROUND(AVG(t.total_amount), 2) AS avg_total
    FROM hw3_taxi_trips AS t
         CROSS JOIN stats AS s
    WHERE ABS(t.total_amount - s.mean_value) > 3 * s.sd_value
    GROUP BY t.ratecode_id
    ORDER BY n_outliers DESC;
""")

**Висновок.** Поріг 3σ — приблизно 97 $. Найбільше «викидів» дає тариф JFK (1 116), далі звичайний тариф (401)
і домовлений (370). Це просто дорогі поїздки (аеропорти, за місто), а не помилки.

#### Доменна перевірка: від'ємні суми або total > 1000

In [ ]:
show("""
    SELECT
        trip_id,
        vendor_id,
        pickup_at AT TIME ZONE 'America/New_York' AS pickup_ny,
        payment_type,
        trip_distance,
        fare_amount,
        tip_amount,
        total_amount
    FROM hw3_taxi_trips
    WHERE fare_amount < 0
       OR tip_amount < 0
       OR total_amount > 1000
    ORDER BY pickup_at
    LIMIT 20;
""")

**Висновок.** Сум понад 1000 $ немає. Усі знайдені рядки мають від'ємні суми — це повернення коштів,
найчастіше зі спором (`payment_type = 4`) або готівкою.

#### Від'ємні суми за payment_type

In [ ]:
show("""
    SELECT
        payment_type,
        COUNT(*) AS n_negative
    FROM hw3_taxi_trips
    WHERE fare_amount < 0 OR total_amount < 0
    GROUP BY payment_type
    ORDER BY n_negative DESC;
""")

**Висновок.** Рядків з від'ємною сумою 405: 247 — спір, 88 — готівка, 49 — без оплати, 19 — картка, 2 — Flex Fare.

#### Що робити зі знайденими аномаліями

| аномалія | дія |
|---|---|
| дорогі аеропортні й міжміські поїздки (3σ) | **залишити** як рідкісний, але нормальний випадок; тариф і зону використати як ознаки |
| від'ємні суми (повернення коштів) | **прапорець** `flag_negative_amount`, **виключити з training data** для прогнозу ціни |
| нульова дистанція, `dropoff = pickup` (vendor 7), `passenger_count = 0` | **прапорець**, **виключити з training data** |
| `total_amount` ≠ сума складових | **виправити за правилом** окремо для кожного `vendor_id` / `payment_type` |
| дублікати vendor 1 (картка + Flex), дорогий домовлений тариф без поїздки | **передати на перевірку джерела** |

Великі значення самі по собі не видаляю.

## Завдання 5. DQL / EDA-запити

Якщо не сказано інше, беруться лише поїздки з `is_valid = TRUE`.

### Q1

**Business-question:** У які години доби (NY local) найбільше поїздок?

In [ ]:
show("""
    SELECT
        pickup_hour,
        COUNT(*) AS n_trips,
        ROUND(AVG(total_amount), 2) AS avg_total,
        ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph
    FROM hw3_taxi_trips
    WHERE is_valid
    GROUP BY pickup_hour
    ORDER BY n_trips DESC, pickup_hour
    LIMIT 5;
""")

**Interpretation:** Найбільше поїздок о 18:00 (5 816) і 17:00 (5 784), далі 21:00, 20:00 і 14:00 — попит вищий у другій половині дня.
О 14–18 год середня швидкість лише 9–10 mph, а о 20–21 год — уже близько 11.7 mph, тому година посадки
впливає на тривалість поїздки і стане в пригоді як ознака.

### Q2

**Business-question:** Яка частка поїздок має NULL passenger_count за типом оплати?

In [ ]:
show("""
    SELECT
        payment_type,
        COUNT(*) AS n_trips,
        COUNT(*) FILTER (WHERE passenger_count IS NULL) AS n_null_passengers,
        ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / COUNT(*), 2) AS pct_null_passengers
    FROM hw3_taxi_trips
    GROUP BY payment_type
    ORDER BY n_trips DESC;
""")

**Interpretation:** `passenger_count` порожній у 100 % поїздок Flex Fare (`payment_type = 0`) і не порожній у всіх інших.
Отже, пропуск залежить від типу запису, а не випадковий: краще додати ознаку «Flex Fare», ніж заповнювати пропуски.

### Q3

**Business-question:** Які поїздки мають від'ємні суми або нульову відстань, хоча оплата не є спірною/анульованою?

In [ ]:
show("""
    SELECT
        trip_id,
        vendor_id,
        payment_type,
        ratecode_id,
        trip_distance,
        trip_duration_min,
        fare_amount,
        total_amount
    FROM hw3_taxi_trips
    WHERE (fare_amount < 0 OR total_amount < 0 OR trip_distance = 0)
      AND NOT (payment_type IN (3, 4, 6))
      AND ratecode_id IS NOT NULL
    ORDER BY total_amount, trip_id
    LIMIT 10;
""")

**Interpretation:** Навіть без спорів і анулювань є багато від'ємних сум, і всі десять перших — оплата готівкою,
здебільшого з тарифом JFK (−70 $, фіксована ціна до аеропорту), і в багатьох є реальна дистанція 12–27 миль.
Отже, це повернення коштів за справжні поїздки, а не окремі поїздки. Залишаю їх з прапорцем, але не використовую для навчання моделі ціни.

### Q4

**Business-question:** Скільки поїздок починається в аеропортах і скільки вони коштують?

In [ ]:
show("""
    SELECT
        z.zone,
        COUNT(*) AS n_trips,
        ROUND(AVG(t.trip_distance), 2) AS avg_distance,
        ROUND(AVG(t.total_amount), 2) AS avg_total
    FROM hw3_taxi_trips AS t
         JOIN hw3_taxi_zones AS z
              ON z.location_id = t.pu_location_id
    WHERE t.is_valid
      AND z.zone ILIKE '%airport%'
    GROUP BY z.zone
    ORDER BY n_trips DESC;
""")

**Interpretation:** З JFK почалося 4 047 валідних поїздок (≈ 4.4 %), з LaGuardia — 2 393, середній чек — 77.50 $ і 70.49 $,
тобто в кілька разів більше за типову поїздку. З Newark лише 6 поїздок із середньою дистанцією 0.05 милі та чеком 138 $ —
схоже на фіксований тариф без даних про поїздку, такі рядки варто перевірити.

### Q5

**Business-question:** Яка частка поїздок з кожного боро завершується в іншому боро?

In [ ]:
show("""
    SELECT
        COALESCE(pu.borough, 'Невідомо / поза NYC') AS pickup_borough,
        COUNT(*) AS n_trips,
        COUNT(*) FILTER (WHERE pu.borough IS DISTINCT FROM dz.borough) AS n_inter_borough,
        ROUND(COUNT(*) FILTER (WHERE pu.borough IS DISTINCT FROM dz.borough) * 100.0 / COUNT(*), 2)
            AS pct_inter_borough
    FROM hw3_taxi_trips AS t
         JOIN hw3_taxi_zones AS pu ON pu.location_id = t.pu_location_id
         JOIN hw3_taxi_zones AS dz ON dz.location_id = t.do_location_id
    WHERE t.is_valid
    GROUP BY pu.borough
    ORDER BY n_trips DESC;
""")

**Interpretation:** 87.8 % валідних поїздок починаються в Manhattan, і лише 7.7 % з них закінчуються в іншому боро.
З Queens навпаки — 72 % поїздок їдуть в інше боро (це аеропорти → Manhattan). `IS DISTINCT FROM` рахує
зони без боро (`NULL`) як інший район, а звичайне `<>` такі рядки пропустило б.

### Q6

**Business-question:** Які комбінації vendor_id і request_source трапляються в даних?

In [ ]:
show("""
    SELECT DISTINCT
        vendor_id,
        request_source
    FROM hw3_taxi_trips
    ORDER BY vendor_id, request_source;
""")

**Interpretation:** Vendor 1 і 2 використовують кілька кодів `request_source`, vendor 6 — лише `A`, а vendor 7 це поле не заповнює взагалі.
Поле заповнюється непослідовно і без опису значень, тому як ознаку я б його поки не використовував.

### Q7

**Business-question:** Наскільки повільніші поїздки у вечірній пік (16–19) порівняно з ніччю (0–5)?

In [ ]:
show("""
    SELECT
        CASE
            WHEN pickup_hour BETWEEN 16 AND 19 THEN 'evening peak 16-19'
            WHEN pickup_hour BETWEEN 0 AND 5 THEN 'night 0-5'
        END AS period,
        COUNT(*) AS n_trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(trip_duration_min), 2) AS avg_duration_min,
        ROUND(AVG(avg_speed_mph), 2) AS avg_speed_mph
    FROM hw3_taxi_trips
    WHERE is_valid
      AND (pickup_hour BETWEEN 16 AND 19 OR pickup_hour BETWEEN 0 AND 5)
    GROUP BY 1
    ORDER BY avg_speed_mph;
""")

**Interpretation:** У вечірній пік середня швидкість 9.7 mph, а вночі — 16.0 mph, тобто на ~40 % менше.
Нічні поїздки довші (4.64 проти 3.17 милі), але займають навіть менше часу (15.6 проти 17.4 хв),
тож тривалість не можна прогнозувати лише за дистанцією.

### Q8

**Business-question:** Який середній відсоток чайових (оплата карткою) у різних боро посадки?

In [ ]:
show("""
    SELECT
        COALESCE(z.borough, 'Невідомо / поза NYC') AS pickup_borough,
        COUNT(*) AS n_card_trips,
        ROUND(AVG(t.tip_amount * 100.0 / NULLIF(t.fare_amount, 0)), 2) AS avg_tip_pct,
        ROUND(COUNT(*) FILTER (WHERE t.tip_amount = 0) * 100.0 / COUNT(*), 2) AS pct_no_tip
    FROM hw3_taxi_trips AS t
         JOIN hw3_taxi_zones AS z
              ON z.location_id = t.pu_location_id
    WHERE t.is_valid
      AND t.payment_type = 1
    GROUP BY z.borough
    HAVING COUNT(*) >= 100
    ORDER BY avg_tip_pct DESC;
""")

**Interpretation:** Найвищий відсоток чайових — у Brooklyn (28.7 %) і Manhattan (26.4 %), найнижчий — у Queens (21.3 %),
де багато аеропортних поїздок із фіксованою ціною. Водночас у Brooklyn удвічі частіше взагалі не лишають чайових
(12.2 % проти 5.5 % у Manhattan). Bronx і Staten Island не показані через `HAVING` — там замало карткових поїздок.

### Q9

**Business-question:** Які маршрути між зонами найпопулярніші?

In [ ]:
show("""
    SELECT
        pu.zone AS pickup_zone,
        dz.zone AS dropoff_zone,
        COUNT(*) AS n_trips,
        ROUND(AVG(t.trip_duration_min), 2) AS avg_duration_min,
        ROUND(AVG(t.total_amount), 2) AS avg_total
    FROM hw3_taxi_trips AS t
         JOIN hw3_taxi_zones AS pu ON pu.location_id = t.pu_location_id
         JOIN hw3_taxi_zones AS dz ON dz.location_id = t.do_location_id
    WHERE t.is_valid
    GROUP BY pu.zone, dz.zone
    ORDER BY n_trips DESC, pickup_zone
    LIMIT 10;
""")

**Interpretation:** Майже всі топ-маршрути — короткі поїздки всередині Manhattan (Upper East Side, Midtown, Upper West Side):
4–13 хвилин і 13–23 $. Найпопулярніший — Upper East Side South → North (491 поїздка) і назад (369).
Виняток — JFK → Outside of NYC (216 поїздок, 43 хв, 123 $).

### Q10

**Business-question:** Як змінюється кількість поїздок за днем тижня (1 = Пн … 7 = Нд)?

In [ ]:
show("""
    SELECT
        pickup_dow,
        COUNT(*) AS n_trips,
        COUNT(DISTINCT (pickup_at AT TIME ZONE 'America/New_York')::DATE) AS n_days,
        ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT (pickup_at AT TIME ZONE 'America/New_York')::DATE), 0) AS avg_trips_per_day,
        ROUND(AVG(trip_distance), 2) AS avg_distance
    FROM hw3_taxi_trips
    WHERE is_valid
    GROUP BY pickup_dow
    ORDER BY pickup_dow;
""")

**Interpretation:** У липні 2026 середа, четвер і п'ятниця трапляються по 5 разів, а решта днів — по 4,
тому порівнюю середню кількість поїздок **за день**. Найбільше — у четвер (3 209) і середу (3 132),
найменше — у понеділок (2 709). У неділю поїздки найдовші (4.12 милі), ймовірно, через поїздки в аеропорти.

### Q11

**Business-question:** Яка частка поїздок платить CBD congestion fee у Manhattan та поза ним?

In [ ]:
show("""
    SELECT
        CASE WHEN z.borough LIKE 'Manhattan%' THEN 'Manhattan' ELSE 'інші / невідомо' END AS pickup_area,
        COUNT(*) AS n_trips,
        ROUND(COUNT(*) FILTER (WHERE COALESCE(t.cbd_congestion_fee, 0) > 0) * 100.0 / COUNT(*), 2) AS pct_cbd_fee,
        ROUND(COUNT(*) FILTER (WHERE COALESCE(t.airport_fee, 0) > 0) * 100.0 / COUNT(*), 2) AS pct_airport_fee
    FROM hw3_taxi_trips AS t
         JOIN hw3_taxi_zones AS z
              ON z.location_id = t.pu_location_id
    WHERE t.is_valid
    GROUP BY 1
    ORDER BY n_trips DESC;
""")

**Interpretation:** CBD congestion fee сплачують 82.6 % поїздок з посадкою в Manhattan і 38.9 % поїздок з інших районів
(ті, що заїжджають у платну зону). Серед поїздок не з Manhattan 56.9 % мають `airport_fee` — отже, це переважно аеропорти.

### Q12

**Business-question:** Які числові ознаки найсильніше пов'язані з fare_amount?

In [ ]:
show("""
    SELECT
        ROUND(CORR(trip_distance, fare_amount)::NUMERIC, 3)     AS distance_vs_fare,
        ROUND(CORR(trip_duration_min, fare_amount)::NUMERIC, 3) AS duration_vs_fare,
        ROUND(CORR(pickup_hour, fare_amount)::NUMERIC, 3)       AS hour_vs_fare,
        ROUND(CORR(passenger_count, fare_amount)::NUMERIC, 3)   AS passengers_vs_fare
    FROM hw3_taxi_trips
    WHERE is_valid;
""")

**Interpretation:** Найсильніше з `fare_amount` пов'язані дистанція (r = 0.88) і тривалість (r = 0.80).
Година посадки майже не має лінійного зв'язку з ціною (r = −0.02), хоча впливає на швидкість, а кількість пасажирів — слабкий (r = 0.10).

## Завдання 6. Reflection

**Проблеми якості.** Текстових маркерів типу `?` немає, але пропуски є в іншому вигляді: у записах Flex Fare (27.6 %)
одночасно порожні `passenger_count`, `ratecode_id`, `store_and_fwd` і частина зборів. «Невідомо» закодовано числами
(`RatecodeID = 99`, зона 264). Точних дублікатів немає, але 41 рядок vendor 1 записано двічі — як картка і як Flex Fare.
Є нульова дистанція (3.7 %), `dropoff = pickup` у всіх записах vendor 7, `passenger_count = 0` і від'ємні суми.
Також `total_amount` часто не збігається із сумою складових, бо перевізники по-різному записують збори.

**Ознаки для ML.** Для прогнозу ціни в момент посадки я б узяв зони посадки й висадки (як категорії), годину і день тижня
за часом NY, `ratecode_id`, `vendor_id` і очікувану дистанцію маршруту. Не брав би `request_source` (72 % пропусків
і невідоме значення кодів) та технічне поле `store_and_fwd`.

**Leakage.** `total_amount`, `tip_amount`, `tolls_amount`, `extra`, `dropoff_at`, фактична тривалість і `payment_type`
стають відомі тільки після поїздки, тому для прогнозу на старті поїздки їх використовувати не можна.

**Пропозиції для data engineer.** Додати ID поїздки і зв'язок повернення коштів з оригінальним записом, описати нові коди
(`request_source`), заповнювати поля у Flex Fare, записувати збори однаково для всіх перевізників
і не приймати записи з `dropoff <= pickup`.

**ML-задача.** Найприродніше — регресія: прогноз ціни або тривалості поїздки. Якщо агрегувати дані за зоною й годиною,
можна прогнозувати попит (forecasting). Класифікація «чи залишить пасажир чайові» можлива лише для оплати карткою,
бо для готівки чайові не записуються.

## Очищення (опційно)

Видалення всіх колонок, доданих під час виконання ноутбука, та зупинка pgserver.

In [ ]:
# run("""
#     ALTER TABLE hw3_taxi_trips
#         DROP COLUMN IF EXISTS trip_duration_min,
#         DROP COLUMN IF EXISTS pickup_hour,
#         DROP COLUMN IF EXISTS pickup_dow,
#         DROP COLUMN IF EXISTS avg_speed_mph,
#         DROP COLUMN IF EXISTS flag_outside_month,
#         DROP COLUMN IF EXISTS flag_zero_duration,
#         DROP COLUMN IF EXISTS flag_long_duration,
#         DROP COLUMN IF EXISTS flag_zero_distance,
#         DROP COLUMN IF EXISTS flag_negative_amount,
#         DROP COLUMN IF EXISTS flag_zero_passengers,
#         DROP COLUMN IF EXISTS flag_unknown_code,
#         DROP COLUMN IF EXISTS flag_too_fast,
#         DROP COLUMN IF EXISTS is_valid;
# """)
# engine.dispose()
# pg.cleanup()